Evaluate whether converting PDFs to Markdown preserves enough document
structure (headings, lists, tables, formulas and hierarchy) to simplify
semantic chunking for the RAG pipeline.

## Environment Setup

Install and import candidate Markdown extraction libraries.

In [ ]:
import fitz
import pymupdf4llm
from tqdm import tqdm

## Load Test Document

Use the same mathematics PDF throughout the experiment.

In [33]:
pdf = []

reader = fitz.open("./src/pdfs/bom.pdf")

with tqdm(total=len(reader), desc="Converting PDF", unit="page") as pbar:
    for page_num in range(len(reader)):
        md = pymupdf4llm.to_markdown(
            reader,
            pages=[page_num],      # Convert only one page
            show_progress=False,   # Disable built-in progress
            use_ocr=False,
        )

        pdf.append(md)
        pbar.update(1)

markdown = "\n".join(pdf)

Converting PDF: 100%|██████████| 224/224 [01:06<00:00,  3.37page/s]


## Markdown Inspection

Manually inspect the generated Markdown to evaluate structural preservation.

In [34]:
print(pdf[0:3])

['# <mark>Basics of Mathematics</mark> \n\n|Sr. No.|Topic name|Page number|\n|---|---|---|\n|1.|Qualitative and Quantitative Analysis|2|\n|2.|Number and Number System, problems on Numbers|7|\n|3.|Fractions, Decimals and Numbers|32|\n|4.|Divisibility and Divisibility Test|39|\n|5.|HCF and LCM|44|\n|6.|Squares, Cubes, Square-roots & Cube-roots|51|\n|7.|Simplifications|58|\n|8.|Ratio and Proportion|69|\n|9.|Chain rule|74|\n|10.|Averages|82|\n|11.|Percentages|91|\n|12.|Profit and Loss|101|\n|13.|Simple and Compound Interest|110|\n|14.|Heights, Distances & Basic Trigonometry|135|\n|15.|Time, Work and Wages|142|\n|16.|Time and Distances|162|\n|17.|Trains, Boats and Streams|176|\n|18.|Clock|197|\n|19.|Series|205|\n|20.|Problems on Ages|210|\n|21.|Discounts – its types and Analysis|216|\n\n\n\n', "# **Section A** \n\n## **A.1 Introduction to Qualitative and** \n\n## **Quantitative Analysis** \n\n### **Qualitative Data Analysis** \n\nQualitative data analysis (QDA) is the process of organizing,

In [35]:
print(markdown)

# <mark>Basics of Mathematics</mark> 

|Sr. No.|Topic name|Page number|
|---|---|---|
|1.|Qualitative and Quantitative Analysis|2|
|2.|Number and Number System, problems on Numbers|7|
|3.|Fractions, Decimals and Numbers|32|
|4.|Divisibility and Divisibility Test|39|
|5.|HCF and LCM|44|
|6.|Squares, Cubes, Square-roots & Cube-roots|51|
|7.|Simplifications|58|
|8.|Ratio and Proportion|69|
|9.|Chain rule|74|
|10.|Averages|82|
|11.|Percentages|91|
|12.|Profit and Loss|101|
|13.|Simple and Compound Interest|110|
|14.|Heights, Distances & Basic Trigonometry|135|
|15.|Time, Work and Wages|142|
|16.|Time and Distances|162|
|17.|Trains, Boats and Streams|176|
|18.|Clock|197|
|19.|Series|205|
|20.|Problems on Ages|210|
|21.|Discounts – its types and Analysis|216|




# **Section A** 

## **A.1 Introduction to Qualitative and** 

## **Quantitative Analysis** 

### **Qualitative Data Analysis** 

Qualitative data analysis (QDA) is the process of organizing, analyzing, and interpreting qualitative 

## Evalution

| Property      | Result | Notes |
| ------------- | ------ | ----- |
| Headings      | ✅      |       |
| Lists         | ✅      |       |
| Tables        | ⚠️     |       |
| Formulas      | ✅      |       |
| Reading Order | ✅      |       |
| Images        | ❌      |       |


## Algorithm

```Use structure as a separation
    Each pdf has different heading styles and size. and PDF reader pymupdf4llm reads it and convert it into the markdown. 
    due to variation of writing style it heading hierarchy marker may vary from pdf to pdf.

    but the writing style of the author remains same throughout the PDF. so the PDF gets same hierarchy marker to same level of heading and list.

    so rather than hardcoding. 
    we identify the structure and we group them into the semantic structure.


Tasks
    1) read PDF (if page wise -> more batter)

    2) Identify the structure
        Focus on heading hierarchy. because topics mostly contains the semantic boundaries as a headings of next topic.


In [40]:
import re

def read_structure(pdf: str) -> list[str]:
    """
    Returns the heading markers used in the markdown.
    """

    pattern = r'^(#{1,6})\s+.+$'

    markers = re.findall(pattern, pdf, flags=re.MULTILINE)

    return sorted(set(markers), key=len)

heading_markers = read_structure(pdf=markdown)

In [44]:
print(heading_markers)

['#', '##', '###']


In [65]:
def map_heading_markers(markers) -> list:
    """
    Map extracted markers to the standard markers.
    To normalize the PDF with standard markers
    """
    standard_markers = [("#"*(i+1)) for i in range(len(markers))] 
    return standard_markers


standard_markers = map_heading_markers(markers=heading_markers)

In [66]:
import re


def normalize_heading(pdf: str, standard_markers: list[dict], heading_markers) -> str:
    """
    Normalize the entire document with normalized markers.
    to do use heading_markers_list. it has mapped the "pdf markers" with "normalized_marker"
    """

    for i in range(len(heading_markers)):
        pdf.replace(heading_markers[i], standard_markers[i])

    return pdf


normalized_text = normalize_heading(pdf=markdown, standard_markers=standard_markers, heading_markers=heading_markers)

normalized_text

"# <mark>Basics of Mathematics</mark> \n\n|Sr. No.|Topic name|Page number|\n|---|---|---|\n|1.|Qualitative and Quantitative Analysis|2|\n|2.|Number and Number System, problems on Numbers|7|\n|3.|Fractions, Decimals and Numbers|32|\n|4.|Divisibility and Divisibility Test|39|\n|5.|HCF and LCM|44|\n|6.|Squares, Cubes, Square-roots & Cube-roots|51|\n|7.|Simplifications|58|\n|8.|Ratio and Proportion|69|\n|9.|Chain rule|74|\n|10.|Averages|82|\n|11.|Percentages|91|\n|12.|Profit and Loss|101|\n|13.|Simple and Compound Interest|110|\n|14.|Heights, Distances & Basic Trigonometry|135|\n|15.|Time, Work and Wages|142|\n|16.|Time and Distances|162|\n|17.|Trains, Boats and Streams|176|\n|18.|Clock|197|\n|19.|Series|205|\n|20.|Problems on Ages|210|\n|21.|Discounts – its types and Analysis|216|\n\n\n\n\n# **Section A** \n\n## **A.1 Introduction to Qualitative and** \n\n## **Quantitative Analysis** \n\n### **Qualitative Data Analysis** \n\nQualitative data analysis (QDA) is the process of organizing, an

In [70]:
import re


def split_by_marker(text: str, marker: str) -> list[str]:
    """
    Splits markdown by a heading marker while keeping the heading
    attached to each section.
    """
    pattern = rf'(?=^{re.escape(marker)}\s+)'

    return [
        section.strip()
        for section in re.split(pattern, text, flags=re.MULTILINE)
        if section.strip()
    ]


# def build_tree(text: str, markers: list[str], level: int = 0) -> dict:
#     """
#     Builds a semantic tree from markdown.

#     Parameters
#     ----------
#     text : str
#         Markdown to parse.

#     markers : list[str]
#         Ordered heading markers.
#         Example:
#         ["#", "##", "###"]

#     level : int
#         Current recursion level.

#     Returns
#     -------
#     dict
#         Semantic node.
#     """

#     node = {
#         "text": text.strip(),
#         "level": level,
#         "children": []
#     }

#     # Base case
#     if level >= len(markers):
#         return node

#     marker = markers[level]

#     sections = split_by_marker(text, marker)

#     # If no headings of this level exist, keep searching deeper.
#     if len(sections) == 1:
#         return build_tree(text, markers, level + 1)

#     # Everything before the first heading belongs to this node.
#     if not sections[0].startswith(marker):
#         node["text"] = sections[0]
#         sections = sections[1:]
#     else:
#         node["text"] = ""

#     for section in sections:
#         node["children"].append(
#             build_tree(section, markers, level + 1)
#         )

#     return node



def build_tree(
    text: str,
    markers: list[str],
    marker_index: int = 0,
    depth: int = 0,
):
    node = {
        "text": text.strip(),
        "depth": depth,
        "children": []
    }

    if marker_index >= len(markers):
        return node

    marker = markers[marker_index]
    sections = split_by_marker(text, marker)

    if len(sections) == 1:
        return build_tree(
            text,
            markers,
            marker_index + 1,
            depth,
        )

    if not sections[0].startswith(marker):
        node["text"] = sections[0]
        sections = sections[1:]
    else:
        node["text"] = ""

    for section in sections:
        node["children"].append(
            build_tree(
                section,
                markers,
                marker_index + 1,
                depth + 1,
            )
        )

    return node

In [71]:
build_tree(text=markdown, markers=standard_markers)

{'text': '',
 'depth': 0,
 'children': [{'text': '# <mark>Basics of Mathematics</mark> \n\n|Sr. No.|Topic name|Page number|\n|---|---|---|\n|1.|Qualitative and Quantitative Analysis|2|\n|2.|Number and Number System, problems on Numbers|7|\n|3.|Fractions, Decimals and Numbers|32|\n|4.|Divisibility and Divisibility Test|39|\n|5.|HCF and LCM|44|\n|6.|Squares, Cubes, Square-roots & Cube-roots|51|\n|7.|Simplifications|58|\n|8.|Ratio and Proportion|69|\n|9.|Chain rule|74|\n|10.|Averages|82|\n|11.|Percentages|91|\n|12.|Profit and Loss|101|\n|13.|Simple and Compound Interest|110|\n|14.|Heights, Distances & Basic Trigonometry|135|\n|15.|Time, Work and Wages|142|\n|16.|Time and Distances|162|\n|17.|Trains, Boats and Streams|176|\n|18.|Clock|197|\n|19.|Series|205|\n|20.|Problems on Ages|210|\n|21.|Discounts – its types and Analysis|216|',
   'depth': 1,
   'children': []},
  {'text': '# **Section A**',
   'depth': 1,
   'children': [{'text': '## **A.1 Introduction to Qualitative and**',
     'dep